In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType, LongType
)

from pyspark.sql.functions import (
    current_timestamp, date_format, col
)


In [0]:
schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)
])

In [0]:
files = [
    "/Volumes/pysparkdbt_taxiproject/source/volumes/nov_data/yellow_tripdata_2025-11.parquet",
    "/Volumes/pysparkdbt_taxiproject/source/volumes/taxi_data/yellow_tripdata_2025-12.parquet"
]

dfs = []
for path in files:
    d = spark.read.format("parquet").schema(schema).load(path)
    dfs.append(d)

from functools import reduce
df = reduce(lambda a, b: a.unionByName(b), dfs)


df = df.withColumn("ingestion_timestamp", current_timestamp())
df = df.withColumn("year_month", date_format(col("tpep_pickup_datetime"), "yyyy-MM"))


df.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("year_month") \
    .saveAsTable("pysparkdbt_taxiproject.bronze.bronze_yellowtaxi")

In [0]:
print("total rows:", df.count())

In [0]:
df.printSchema()   
df.groupBy("year_month").count().orderBy("year_month").show()